# ベクトル DB に保存して、メタデータで絞り込んで検索する (Qdrant)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_rag/rag_vector_db.ipynb)

NLP クックブック 21-04 ／ RAG（検索拡張生成）

埋め込みを毎回計算し直さずに保存しておきたい。「この部署の文書だけ」「今年の文書だけ」のように条件で絞り込んで検索したい。

In [1]:
!pip install -q qdrant-client

In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
docs = [
    {"text": "有給休暇は入社 6 か月後に 10 日付与されます。", "dept": "人事", "year": 2025},
    {"text": "有給休暇は入社時に 5 日、6 か月後にさらに 10 日付与されます。", "dept": "人事", "year": 2026},
    {"text": "経費精算は月末までに申請システムから行ってください。", "dept": "経理", "year": 2026},
    {"text": "出張旅費の上限は 1 泊あたり 12,000 円です。", "dept": "経理", "year": 2025},
    {"text": "出張旅費の上限は 1 泊あたり 15,000 円に改定されました。", "dept": "経理", "year": 2026},
    {"text": "社内 Wi-Fi のパスワードは四半期ごとに変更されます。", "dept": "情シス", "year": 2026},
    {"text": "ノート PC の貸与期間は原則 4 年です。", "dept": "情シス", "year": 2025},
]

client = QdrantClient(path="qdrant_db")      # ローカルのフォルダに保存 (サーバー不要)。":memory:" ならメモリ上のみ
if not client.collection_exists("manual"):
    client.create_collection("manual", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
vectors = model.encode(["passage: " + d["text"] for d in docs], normalize_embeddings=True)
client.upsert("manual", points=[models.PointStruct(id=i, vector=v.tolist(), payload=d) for i, (v, d) in enumerate(zip(vectors, docs))])
print(client.count("manual"))

count=7


In [3]:
def search(query, limit=3, **conditions):
    must = [models.FieldCondition(key=k, match=models.MatchValue(value=v)) for k, v in conditions.items()]
    result = client.query_points("manual", query=model.encode("query: " + query, normalize_embeddings=True).tolist(),
                                 limit=limit, query_filter=models.Filter(must=must) if must else None)
    return [(round(p.score, 3), p.payload["dept"], p.payload["year"], p.payload["text"]) for p in result.points]

print("絞り込みなし:")
for r in search("出張のホテル代はいくらまで？"):
    print("  ", r)
print("2026 年の文書だけ:")
for r in search("出張のホテル代はいくらまで？", year=2026):
    print("  ", r)
print("人事の文書だけ:")
for r in search("休みは何日もらえる？", dept="人事", limit=2):
    print("  ", r)

絞り込みなし:
   (0.872, '経理', 2025, '出張旅費の上限は 1 泊あたり 12,000 円です。')
   (0.869, '経理', 2026, '出張旅費の上限は 1 泊あたり 15,000 円に改定されました。')
   (0.795, '経理', 2026, '経費精算は月末までに申請システムから行ってください。')
2026 年の文書だけ:
   (0.869, '経理', 2026, '出張旅費の上限は 1 泊あたり 15,000 円に改定されました。')
   (0.795, '経理', 2026, '経費精算は月末までに申請システムから行ってください。')
   (0.784, '人事', 2026, '有給休暇は入社時に 5 日、6 か月後にさらに 10 日付与されます。')
人事の文書だけ:
   (0.848, '人事', 2026, '有給休暇は入社時に 5 日、6 か月後にさらに 10 日付与されます。')
   (0.84, '人事', 2025, '有給休暇は入社 6 か月後に 10 日付与されます。')


In [4]:
# 開き直しても残っている (永続化)。文書の更新は同じ ID で upsert、削除は delete
client.close()
client = QdrantClient(path="qdrant_db")
print(client.count("manual").count, "件")
client.delete("manual", points_selector=models.PointIdsList(points=[3]))   # 古い旅費規程を削除
print([r[3] for r in search("出張のホテル代はいくらまで？", limit=2)])

7 件
['出張旅費の上限は 1 泊あたり 15,000 円に改定されました。', '経費精算は月末までに申請システムから行ってください。']


- 古い版と新しい版の文書が両方ヒットすると、LLM は古い情報で答えてしまうことがある。日付や版でのフィルタ、古い文書の削除が実用上とても重要。
- 数万件までなら NumPy や FAISS で十分。ベクトル DB の利点は、永続化、メタデータでの絞り込み、文書の追加・更新・削除が簡単なこと。
- Qdrant のほかに Chroma、LanceDB、pgvector (PostgreSQL) などがある。ローカルのファイルで動くものから始めると手軽。